<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/10_interpreters_ptc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10 · Interpreters and programmatic tool calling

Lesson 09 gave the agent a computer to act **on**. This one gives it a computer to think **in**.

An interpreter is a JavaScript runtime that lives inside the agent loop. The agent writes code,
the code runs, and only the last expression comes back. Turn on **programmatic tool calling
(PTC)** and your tools become functions that code can call — in loops, in parallel, with the
results filtered *before* the model ever sees them.

The reason to care is not speed. It is that the interpreter is a **wall**, and you get to decide
what crosses it.

**New in this lesson:** `CodeInterpreterMiddleware`, the `eval` tool, `ptc=[...]`

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.
>
> Running locally instead? `uv sync`, then put the key in a `.env` at the repo root.

In [ ]:
# --- snippet:setup v2 ---
import contextlib
import os

try:
    from google.colab import userdata

    %pip install -qq --progress-bar off \
      "deepagents~=0.7.6" \
      "langchain~=1.3.15" \
      "langchain-openai~=1.5.1" \
      "langsmith~=0.11.2" \
      "langchain-quickjs~=0.3.5"

    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    with contextlib.suppress(Exception):
        os.environ["LANGSMITH_GATEWAY_API_KEY"] = userdata.get("LANGSMITH_GATEWAY_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

key = os.environ["LANGSMITH_API_KEY"]
os.environ["LANGSMITH_GATEWAY_API_KEY"] = os.environ.get("LANGSMITH_GATEWAY_API_KEY") or key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-10-interpreters-ptc"

MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. The interpreter on its own

`CodeInterpreterMiddleware` adds one tool, `eval`. Code runs in **QuickJS** with no filesystem,
no network, no shell, no clock. It can compute, hold variables between calls, and log. That is
the entire surface.

Which sounds useless until you notice how much agent work is arithmetic and reshaping that a
model does slowly, expensively, and sometimes wrongly.

In [ ]:
from deepagents import create_deep_agent
from langchain_quickjs import CodeInterpreterMiddleware

calculator = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You have a JavaScript interpreter, the eval tool. "
        "Never do arithmetic in your head — write code and run it."
    ),
    middleware=[CodeInterpreterMiddleware(mode="turn")],
)
calculator

In [ ]:
result = await calculator.ainvoke({"messages": [{"role": "user", "content":
    "A desk costs $429 and we sold 37 of them, minus a 12% bulk discount on the last 9. "
    "What was the total, and what was the average unit price?"
}]})

print(result["messages"][-1].text)

Look at the trace and you will find the JavaScript it wrote. The number is not a guess — it is
the output of a program you can read.

---

## 2. Back to the support desk

The orders from lessons 03 and 06, unchanged. Six of them, each carrying something an agent has
no business broadcasting: a customer's email address.

In [ ]:
#@title Synthetic support data (run me) { display-mode: "form" }
# Six orders, eight tickets, one refund policy. Small on purpose: you should be able to
# read the whole dataset and judge the agent's answers yourself.

# --- snippet:support_data v1 ---
ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1044", "customer": "avery@example.com", "item": "Monitor arm",    "status": "in_transit", "days_ago": 1,  "price": 89.00},
    {"id": "1045", "customer": "sam@example.com",   "item": "Office chair",   "status": "delivered",  "days_ago": 10, "price": 249.00},
    {"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray",  "status": "cancelled",  "days_ago": 7,  "price": 59.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

TICKETS = [
    {"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."},
    {"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."},
    {"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."},
    {"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"},
    {"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."},
    {"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."},
    {"id": "T-7", "order_id": "1042", "text": "Following up on the cracked desk leg. Any update?"},
    {"id": "T-8", "order_id": "9999", "text": "Order never arrived."},
]

REFUND_POLICY = """
# Refund policy

- Damaged on arrival: full refund or replacement, no time limit. Photos required.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days of delivery. Restocking fee 10%.
- Cancelled orders: refund within 5 business days. Escalate if the customer was charged.
- Refunds above $200 require human approval.
"""
# --- /snippet ---

print(f"{len(ORDERS)} orders, {len(TICKETS)} tickets, {len(REFUND_POLICY.splitlines())} lines of policy")

In [ ]:
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> dict:
    """Look up one order by its 4-digit ID.

    Returns the whole order record, with these exact keys:
    id, customer, item, status, days_ago (days since delivery), price.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return order
    return {"error": f"no order with id {order_id}"}


print("returns whole records, PII and all:", lookup_order.invoke({"order_id": "1042"}))

This tool returns the *whole row*. That is realistic — most internal APIs do — and it is the
problem. Every field it returns is about to become part of a prompt.

---

## 3. The ordinary way

Give the agent the tool directly and ask a question that needs every order.

In [ ]:
direct_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order],
    system_prompt="You are a support analyst. Use lookup_order to check orders before answering.",
)
direct_agent

In [ ]:
QUESTION = (
    "Which of orders 1042, 1043, 1044, 1045, 1046 and 1047 were delivered more than "
    "30 days ago? List just the order id and the item."
)

direct = await direct_agent.ainvoke({"messages": [{"role": "user", "content": QUESTION}]})
print(direct["messages"][-1].text)

Right answer, most likely. Now the part nobody looks at — what is *in* the conversation that
produced it:

In [ ]:
import re


def emails_in_context(result) -> list[str]:
    """Every customer address that ended up in the model's message history."""
    blob = "".join(str(getattr(m, "content", "")) for m in result["messages"])
    return sorted(set(re.findall(r"[\w.+-]+@example\.com", blob)))


print("tool calls :", sum(len(getattr(m, "tool_calls", None) or []) for m in direct["messages"]))
print("emails     :", emails_in_context(direct))

Six lookups, and every customer address they touched is now sitting in the context window —
logged in the trace, re-sent on every subsequent model call, and available to the model to
repeat. (Four addresses, not six: two customers ordered twice.)

Nobody asked for them. The tool volunteered them because the tool returns rows.

> **Checkpoint.** Lesson 06 solved a version of this with `PIIMiddleware`. Why is that not
> enough here? Because redaction is a filter on data that has *already* crossed into the
> context — you have to know the shape of every secret in advance. What follows moves the wall
> instead of filtering at it.

---

## 4. The same question through PTC

One change, and it is a subtraction: `lookup_order` is **not** in `tools=` any more. It is only
on the interpreter's `ptc` allowlist.

The agent can no longer call it. Only code can. Names arrive camelCased, so `lookup_order`
becomes `tools.lookupOrder`, and the argument object still follows the tool's schema.

In [ ]:
ptc_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a support analyst with a JavaScript interpreter, the eval tool.\n"
        "lookup_order is available ONLY inside the interpreter, as "
        "tools.lookupOrder({order_id}).\n"
        "Prefer a single eval script: look up every order, filter in the interpreter, and "
        "return only the fields you were asked for. Never return whole order records."
    ),
    middleware=[CodeInterpreterMiddleware(ptc=[lookup_order], mode="turn", timeout=60.0)],
)
ptc_agent

In [ ]:
ptc = await ptc_agent.ainvoke({"messages": [{"role": "user", "content": QUESTION}]})
print(ptc["messages"][-1].text)

Same answer. Now the same audit:

In [ ]:
print("eval calls :", sum(1 for m in ptc["messages"]
                          for t in (getattr(m, "tool_calls", None) or []) if t["name"] == "eval"))
print("emails     :", emails_in_context(ptc) or "none — the records never crossed the wall")

The lookups still happened. Six of them. But they happened *inside* QuickJS, and the model was
handed only what the script chose to return.

Here is the code it wrote to do that:

In [ ]:
for message in ptc["messages"]:
    for call in (getattr(message, "tool_calls", None) or []):
        if call["name"] == "eval":
            print(next(iter(call["args"].values())))

`Promise.all` over every id, a `filter`, and a `map` down to two fields. Because a loop ran the
lookups, **every** order was checked — no sampling, which is where models quietly fail when you
ask for fifty items instead of six.

You may see more than one `eval` here. The interpreter is a REPL, so an agent that guesses a
field name wrong can print `Object.keys(...)`, look, and retry — cheaply, without the failure
ever reaching your context. That is the schema-discovery loop lesson 03 tried to prevent with a
good docstring, happening in code instead of in the model's head.

> **Checkpoint.** Find both runs in LangSmith and compare input tokens on the final model call.
> With six orders the gap is small. Picture the same shape with 500 orders, each row a paragraph
> of description: the direct agent's context grows with the data, the PTC agent's does not.

---

## 5. A permission boundary, not just a token trick

Most write-ups sell PTC on tokens. The token saving is real and it is the least interesting part.

Passing a tool **only** through `ptc` changes what is reachable:

| | In `tools=` | In `ptc=` only |
|---|---|---|
| Model can call it directly | yes | **no** |
| Raw result enters context | always | only if the script returns it |
| Who decides what is kept | the model, after seeing everything | your prompt, before anything is seen |
| Calls per model turn | one | as many as the code writes |

That last column is also the catch.

### The catch, and it is a sharp one

Lesson 06 wrapped `issue_refund` in `interrupt_on` so a human approves every refund. **PTC calls
do not go through the tool-calling path, so `interrupt_on` is not enforced for them.** A tool on
the `ptc` allowlist can be called in a loop by generated code with no approval step.

So the rule is narrow and worth memorising:

> Put **read** tools behind PTC. Keep anything that spends money, sends a message, or changes
> state in `tools=`, where the interrupt still fires.

`lookup_order` is a read. `issue_refund` would never belong here.

---

## 6. Interpreter or sandbox?

Two lessons, two ways to run code, and they are not alternatives.

| | Interpreter (10) | Sandbox (09) |
|---|---|---|
| Runs | JavaScript in QuickJS, in-process | anything, in a container |
| Reaches | your tools, via `ptc` | filesystem, shell, network, `pip` |
| Costs | nothing to start | a container to create and delete |
| Good at | composing tools, filtering data, deciding what the model sees | doing real work in an environment |
| Wall protects | your **context** from the data | your **host** from the code |

Use the interpreter to shape what the model sees. Use the sandbox to let the model act.

### State between turns

`mode` controls how long interpreter variables live:

- `"call"` — every `eval` starts empty.
- `"turn"` — variables survive across `eval` calls within one turn. What we used.
- `"thread"` (default) — variables survive across turns too, so a follow-up question can reuse
  a corpus the agent already built. Snapshots keep **serializable values only**; a function you
  defined last turn comes back as an error, not a function.

---

## 📌 Key takeaways

- An interpreter is a JavaScript runtime **inside** the agent loop: no filesystem, no network, no shell. It adds one tool, `eval`.
- **PTC** exposes chosen tools to that code as `tools.camelCased(...)`. The agent writes one script instead of one call at a time.
- A tool in `ptc=` but **not** in `tools=` is unreachable by the model directly — the interpreter becomes a permission boundary, not just a context boundary.
- Raw rows stay behind the wall. Only the script's return value becomes context, which is how six customer emails became zero.
- Because a loop drives it, **every** item gets processed. Models sample; code does not.
- **`interrupt_on` is not enforced for PTC calls.** Read tools behind PTC; anything that spends money or changes state stays in `tools=`.
- PTC is async — `await agent.ainvoke(...)`. Interpreters are beta.
- Interpreter protects your context from the data; sandbox protects your host from the code.

---

## ➡️ Next

**[11 · Under the hood](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/11_under_the_hood.ipynb)**

You have now seen every piece of the harness. Time to take the lid off and find out that
`create_deep_agent` is `create_agent` with a well-chosen middleware stack — and that underneath
both is a LangGraph graph you can step through.